# Chapter 1: Spark Fundamentals

## Overview

This notebook introduces the fundamentals of Apache Spark, a unified analytics engine for large-scale data processing.

### Learning Objectives

- Understand Spark architecture and components
- Create and manipulate Spark DataFrames
- Perform basic data transformations
- Understand RDDs vs DataFrames vs Datasets
- Use Spark SQL for data analysis

## Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, when
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

In [ ]:
# Create Spark session
spark = SparkSession.builder \
    .appName("SparkFundamentals") \
    .master("spark://spark-master:7077") \
    .config("spark.executor.memory", "1g") \
    .config("spark.executor.cores", "1") \
    .getOrCreate()

# Verify Spark session
print(f"Spark version: {spark.version}")
print(f"Spark master: {spark.conf.get('spark.master')}")

## Exercise 1: Creating Your First DataFrame

In [ ]:
# Create sample data
data = [
    ("Alice", 25, "Engineering"),
    ("Bob", 30, "Marketing"),
    ("Charlie", 35, "Engineering"),
    ("Diana", 28, "Sales"),
    ("Eve", 32, "Marketing")
]

# Define schema
columns = ["name", "age", "department"]

# Create DataFrame
df = spark.createDataFrame(data, columns)

# Display the DataFrame
df.show()

In [ ]:
# Explore DataFrame schema
df.printSchema()
print(f"\nColumn names: {df.columns}")
print(f"\nRow count: {df.count()}")

## Exercise 2: Basic DataFrame Operations

In [ ]:
# Filter data
print("Employees older than 30:")
df.filter(df.age > 30).show()

# Select columns
print("\nName and Department:")
df.select("name", "department").show()

# Sort data
print("\nSorted by age:")
df.orderBy("age").show()

# Group by and count
print("\nEmployees per department:")
df.groupBy("department").count().show()

## Exercise 3: Adding Columns and Transformations

In [ ]:
# Add a new column
df_with_salary = df.withColumn("salary", col("age") * 2000)
df_with_salary.show()

# Rename column
df_renamed = df_with_salary.withColumnRenamed("age", "years_old")
df_renamed.show()

# Drop column
df_dropped = df_renamed.drop("years_old")
df_dropped.show()

## Exercise 4: Spark SQL

In [ ]:
# Register DataFrame as temporary view
df.createOrReplaceTempView("employees")

# Run SQL query
result = spark.sql("""
    SELECT department,
           COUNT(*) as employee_count,
           AVG(age) as avg_age
    FROM employees
    GROUP BY department
    ORDER BY employee_count DESC
""")

result.show()

## Exercise 5: Caching and Persistence

In [ ]:
import time

# Cache DataFrame
df.cache()

# First access (will compute)
start = time.time()
df.count()
first_time = time.time() - start
print(f"First access (computes): {first_time:.4f} seconds")

# Second access (uses cache)
start = time.time()
df.count()
second_time = time.time() - start
print(f"Second access (cached): {second_time:.4f} seconds")

# Un cache
df.unpersist()

## Cleanup

In [ ]:
# Stop Spark session
spark.stop()
print("Spark session stopped.")